<!-- TARGET: private master. build_m05.py writes the follow-along version to the public repo before class and the full version after class. -->
# Module 5 · Live demo

**Python for Data Science · ESADE MIBA · Wed 11 Nov 2026**

The CFO asks for one number: *total revenue*. We have two candidates, 16.0 M and 20.3 M BRL.
By the end of today you can say which one is right, and prove it.

Two revenue numbers, used the same way in every notebook from now on:

- **Paid revenue** = `payment_value`, summed per order. What customers paid.
- **Item revenue** = `price + freight_value`, summed over item lines. What each item sold for.
  The only one that can be split by seller or category.

In [1]:
from pathlib import Path

import pandas as pd

RAW = Path("../../data/raw")
CLEAN = Path("../../data/clean")

## 0. Raw → clean

`data/raw/` is read-only: what Olist gave us. `data/clean/` holds the same tables, **typed**:
one file per table, written by code, never by hand.

In [6]:
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=DATE_COLUMNS)
CLEAN.mkdir(parents=True, exist_ok=True)
orders.to_parquet(CLEAN / "orders.parquet", index=False)

Why parquet and not CSV? Write the typed table to CSV, read it back, and compare the dtypes.

*Your turn (1 min):* predict the dtype of `order_purchase_timestamp` in each version. Then
read `orders.parquet` back into `orders_from_parquet`.

In [7]:
orders.to_csv(CLEAN / "orders_roundtrip.csv", index=False)
orders_from_csv = pd.read_csv(CLEAN / "orders_roundtrip.csv")
orders_from_parquet = pd.read_parquet(CLEAN / "orders.parquet")

In [8]:
orders_from_csv = pd.read_csv(CLEAN / "orders_roundtrip.csv")
orders_from_parquet = pd.read_parquet(CLEAN / "orders.parquet")
pd.DataFrame({"csv": orders_from_csv.dtypes, "parquet": orders_from_parquet.dtypes})

,csv,parquet
order_id,object,object
customer_id,object,object
order_status,object,object
order_purchase_timestamp,object,datetime64[ns]
order_approved_at,object,datetime64[ns]
order_delivered_carrier_date,object,datetime64[ns]
order_delivered_customer_date,object,datetime64[ns]
order_estimated_delivery_date,object,datetime64[ns]


In [9]:
(CLEAN / "orders_roundtrip.csv").unlink()

CSV is text: every type is lost on the way out. Parquet stores the types, and the file is
smaller. `pyarrow`, in your `pds` env since M01, is the engine that does it.

From here on, the clean layer is read with `pd.read_parquet`.

## 1. Grain: what is one row?

Before any `groupby` or `merge`, say what one row of each table **is**, then test it.

In [10]:
orders = pd.read_parquet(CLEAN / "orders.parquet")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")

### One row of `orders` is one order

In [18]:
assert orders["order_id"].is_unique

### One row of `customers` is… a customer *per order*

In [29]:
customers.groupby("customer_unique_id").count().customer_id#.plot(kind="hist")

customer_unique_id
0000366f3b9a7992bf8c76cfdf3221e2    1
0000b849f77a49e4a4ce2b2a4ca5be3f    1
0000f46a3911fa3c0805444483337064    1
0000f6ccb0745a6a4b88665a16c9f078    1
0004aac84e0df4da2b147fca70cf8255    1
                                   ..
fffcf5a5ff07b0908bd4e2dbc735a684    1
fffea47cd6d3cc0a88bd621562a9d061    1
ffff371b4d645b6ecea244b27531430a    1
ffff5962728ec6157033ef9805bacc48    1
ffffd2657e2aad2907e67c3e9daecbeb    1
Name: customer_id, Length: 96096, dtype: int64

In [39]:
customers.groupby("customer_unique_id").nunique().customer_city.sort_values(ascending=False).head()

customer_unique_id
d44ccec15f5f86d14d6a2cfa67da1975    3
11b47516f8f6a42105b9807903ede333    2
9f67aa18a87ea9a76854deeb9385c011    2
d3f15df5924e63131bcdac6364b6a6fd    2
ec0f5b4fe392d71f361112c1b508b707    2
Name: customer_city, dtype: int64

In [40]:
customers[customers.customer_unique_id == "d44ccec15f5f86d14d6a2cfa67da1975"]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
45851,844a7e5d2737f2f6667a5242486cbcb1,d44ccec15f5f86d14d6a2cfa67da1975,3533,sao paulo,SP
57951,105e31e3b36d904c70058578126c4d12,d44ccec15f5f86d14d6a2cfa67da1975,62800,aracati,CE
83347,488dd91f70d8404b2aa677cdc8a0de68,d44ccec15f5f86d14d6a2cfa67da1975,88371,navegantes,SC


In [19]:
print(len(customers), customers["customer_id"].is_unique)
print(customers["customer_unique_id"].nunique())

99441 True
96096


`customer_id` is created for each order. The person is `customer_unique_id`: 99,441 rows,
96,096 people. "How many customers do we have?" has two answers. Only one of them is a person.

### One row of `items` is one item line of an order

In [35]:
items.duplicated(subset=["order_id", "order_item_id"]).sum()

np.int64(0)

In [30]:
print(items["order_id"].is_unique)
print(items.duplicated(subset=["order_id", "order_item_id"]).any())

False
False


`order_id` alone is not unique: an order can have several items. The pair
`(order_id, order_item_id)` is. That pair is the **key** of the table.

*Your turn (2 min):* what is one row of `payments`? Find its key and prove it: assign to
`payments_key_is_unique` the result of checking that no row repeats the key.

In [57]:
(
    payments
        .groupby("order_id")
        .payment_sequential
        .count()
        .sort_values()
        .tail(10)
)

order_id
4689b1816de42507a7d63a4617383c59    14
3c58bffb70dcf45f12bdf66a3c215905    14
4bfcba9e084f46c8e3cb49b0fa6e6159    15
21577126c19bf11a0b91592e5844ba78    15
fedcd9f7ccdc8cba3a18defedd1a5547    19
ee9ca989fc93ba09a6eddc250ce01742    19
895ab968e7bb0d5659d16cd74cd1650c    21
285c2e15bebd4ac83635ccc563dc71f4    22
ccf804e764ed5650cd8759557269dc13    26
fa65dad1b0e818e3ccc5cb0e39231352    29
Name: payment_sequential, dtype: int64

In [50]:
payments[payments.order_id == "ee9ca989fc93ba09a6eddc250ce01742"].sort_values("payment_sequential")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
8910,ee9ca989fc93ba09a6eddc250ce01742,1,credit_card,1,1.76
52343,ee9ca989fc93ba09a6eddc250ce01742,2,voucher,1,5.88
35428,ee9ca989fc93ba09a6eddc250ce01742,3,voucher,1,12.49
64576,ee9ca989fc93ba09a6eddc250ce01742,4,voucher,1,3.49
68844,ee9ca989fc93ba09a6eddc250ce01742,5,voucher,1,3.58
85552,ee9ca989fc93ba09a6eddc250ce01742,6,voucher,1,3.75
96065,ee9ca989fc93ba09a6eddc250ce01742,7,voucher,1,1.92
2099,ee9ca989fc93ba09a6eddc250ce01742,8,voucher,1,5.31
94178,ee9ca989fc93ba09a6eddc250ce01742,9,voucher,1,3.90
86515,ee9ca989fc93ba09a6eddc250ce01742,10,voucher,1,2.45


In [46]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [51]:
payments.head()
payments_key_is_unique = not payments.duplicated(subset=["order_id", "payment_sequential"]).any()

In [52]:
payments_key_is_unique

True

In [53]:
assert not payments.duplicated(subset=["order_id", "payment_sequential"]).any()
print((payments["order_id"].value_counts() > 1).sum(), "orders paid in more than one payment")

2961 orders paid in more than one payment


### Reviews: one per order?

In [55]:
print(reviews["order_id"].is_unique)
repeated_reviews = reviews[reviews["order_id"].duplicated(keep=False)]
print(repeated_reviews["order_id"].nunique(), "orders with more than one review")
repeated_reviews.sort_values("order_id").head(6)[["order_id", "review_score", "review_creation_date"]]

False
547 orders with more than one review


,order_id,review_score,review_creation_date
25612,0035246a40f520710769010f752e7507,5,2017-08-29 00:00:00
22423,0035246a40f520710769010f752e7507,5,2017-08-25 00:00:00
22779,013056cfe49763c6f66bda03396c5ee3,5,2018-02-22 00:00:00
68633,013056cfe49763c6f66bda03396c5ee3,4,2018-03-04 00:00:00
854,0176a6846bcb3b0d3aa3116a9a768597,5,2017-12-30 00:00:00
83224,0176a6846bcb3b0d3aa3116a9a768597,5,2017-12-30 00:00:00


547 orders have two or more reviews. Any "average score per order" has to decide which one
counts. You make that decision in the lab.

**The rule:** an `assert` turns "I think one row is an order" into a check that fails loudly
the day it stops being true.

In [56]:
assert orders["order_id"].is_unique
assert customers["customer_id"].is_unique
assert not items.duplicated(subset=["order_id", "order_item_id"]).any()

## 2. Split-apply-combine

**Split** the rows into groups, **apply** a function to each group, **combine** the results
into one table. In pandas: `groupby(...).agg(...)`.

### How do customers pay?

In [62]:
results = []
for g, df in payments.groupby("payment_type"):
    total_payment = df.payment_value.sum()
    results.append({"payment_type": g, "payment_value": total_payment})

pd.DataFrame(results)

,payment_type,payment_value
0,boleto,2869361.27
1,credit_card,12542084.19
2,debit_card,217989.79
3,not_defined,0.00
4,voucher,379436.87


In [77]:
(
    payments
        .groupby("payment_type")["payment_value"]
        .sum()
        .reset_index()
        .sort_values("payment_value", ascending=False)
)

,payment_type,payment_value
1,credit_card,12542084.19
0,boleto,2869361.27
4,voucher,379436.87
2,debit_card,217989.79
3,not_defined,0.00


One number per group. Usually the business wants several. **Named aggregations**: each output
column is `name=(column, function)`.

In [72]:
import numpy as np

In [81]:
def my_perc(x):
    return np.percentile(x, 5)

In [ ]:
my_perc = lambda x: np.percentile(x, 5)

In [82]:
by_payment_type = payments.groupby("payment_type").agg(
    paid_revenue=("payment_value", np.sum),
    payments=("payment_value", "size"),
    mean_payment=("payment_value", "mean"),
    mean_installments=("payment_installments", "mean"),
    std_payment=("payment_value", my_perc)
)
by_payment_type.sort_values("paid_revenue", ascending=False)

/tmp/ipykernel_32955/828376334.py:1: FutureWarning: The provided callable <function sum at 0x7f4c90253600> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  by_payment_type = payments.groupby("payment_type").agg(


,paid_revenue,payments,mean_payment,mean_installments,std_payment
payment_type,,,,,
credit_card,12542084.19,76795,163.319021,3.507155,30.230
boleto,2869361.27,19784,145.034435,1.000000,30.793
voucher,379436.87,5775,65.703354,1.000000,3.327
debit_card,217989.79,1529,142.570170,1.000000,28.194
not_defined,0.00,3,0.000000,1.000000,0.000


Credit cards carry three quarters of the revenue and are paid in 3.5 instalments on average.
The other methods always pay in one go.

### `size` vs `count`

In [83]:
orders.groupby("order_status").agg(
    orders=("order_id", "size"),
    with_delivery_date=("order_delivered_customer_date", "count"),
)

,orders,with_delivery_date
order_status,,
approved,2,0
canceled,625,6
created,5,0
delivered,96478,96470
invoiced,314,0
processing,301,0
shipped,1107,0
unavailable,609,0


`size` counts rows. `count` counts **non-missing** values. Same table, two different questions.
The 8 delivered orders without a date from M04 are visible here.

### The top sellers

In [84]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [87]:
items = items.assign(item_revenue=items["price"] + items["freight_value"])

In [88]:
items["item_revenue_2"] = items["price"] + items["freight_value"]

In [90]:
assert (items.item_revenue == items.item_revenue_2).all()

In [91]:
top_sellers = (
    items.groupby("seller_id")
    .agg(
        item_revenue=("item_revenue", "sum"),
        items_sold=("order_item_id", "size"),
        orders=("order_id", "nunique"),
    )
    .sort_values("item_revenue", ascending=False)
)
top_sellers.head(10)

,item_revenue,items_sold,orders
seller_id,,,
4869f7a5dfa277a7dca6462dcf3b52b2,249640.70,1156,1132
7c67e1448b00f6e969d365cea6b010ab,239536.44,1364,982
53243585a1d6dc2643021fd1853d8905,235856.68,410,358
4a3ca9315b744ce9f8e9374361493884,235539.96,1987,1806
fa1c13f2614d7b5c4749cbc52fecda94,204084.73,586,585
da8622b14eb17ae2831f4ac5b9dab84a,185192.32,1551,1314
7e93a43ef30c4f03f38b393420bc753a,182754.05,340,336
1025f0e2d44d7041d6cf58b6550e0bfa,172860.69,1428,915
7a67c85e85bb2ce8582c35f2203ad736,162648.38,1171,1160


`items_sold` and `orders` differ: a seller can ship several items in one order. `nunique`
counts orders, not lines.

*Your turn (3 min):* among sellers with **at least 100 orders**, which ten have the highest
mean item revenue per item sold? Store the ten rows in `premium_sellers`.

In [92]:
busy_sellers = top_sellers[top_sellers["orders"] >= 100]

In [93]:
busy_sellers.head()

,item_revenue,items_sold,orders
seller_id,,,
4869f7a5dfa277a7dca6462dcf3b52b2,249640.70,1156,1132
7c67e1448b00f6e969d365cea6b010ab,239536.44,1364,982
53243585a1d6dc2643021fd1853d8905,235856.68,410,358
4a3ca9315b744ce9f8e9374361493884,235539.96,1987,1806
fa1c13f2614d7b5c4749cbc52fecda94,204084.73,586,585


In [97]:
premium_sellers = (
    busy_sellers
    .assign(revenue_per_item=busy_sellers["item_revenue"] / busy_sellers["items_sold"])
    .sort_values("revenue_per_item", ascending=False)
    .head(10)
)
premium_sellers

,item_revenue,items_sold,orders,revenue_per_item
seller_id,,,,
53243585a1d6dc2643021fd1853d8905,235856.68,410,358,575.260195
7e93a43ef30c4f03f38b393420bc753a,182754.05,340,336,537.511912
edb1ef5e36e0c8cd84eb3c9b003e486d,83189.65,175,166,475.369429
f5a590cf36251cf1162ea35bef76fe84,57440.94,122,117,470.827377
ccc4bbb5f32a6ab2b7066a4130f114e3,79261.60,192,187,412.820833
5dceca129747e92ff8ef7a997dc4f8ca,125816.92,346,325,363.632717
fa1c13f2614d7b5c4749cbc52fecda94,204084.73,586,585,348.267457
522620dcb18a6b31cd7bdf73665113a9,61829.81,180,174,343.498944
17a053fcb14bd219540cbde0df490be0,49531.57,149,125,332.426644


In [95]:
premium_sellers.head()

,item_revenue,items_sold,orders,revenue_per_item
seller_id,,,,
4869f7a5dfa277a7dca6462dcf3b52b2,249640.70,1156,1132,215.952163
7c67e1448b00f6e969d365cea6b010ab,239536.44,1364,982,175.613226
53243585a1d6dc2643021fd1853d8905,235856.68,410,358,575.260195
4a3ca9315b744ce9f8e9374361493884,235539.96,1987,1806,118.540493
fa1c13f2614d7b5c4749cbc52fecda94,204084.73,586,585,348.267457


### Changing the grain on purpose

Payments are at payment grain. Most questions are about orders. Group by `order_id` and you
get **one row per order**: a new table at a new grain.

In [98]:
paid_by_order = payments.groupby("order_id").agg(paid_revenue=("payment_value", "sum")).reset_index()
assert paid_by_order["order_id"].is_unique

In [99]:
paid_by_order.head()

,order_id,paid_revenue
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [101]:
print(len(payments), "payments →", len(paid_by_order), "orders")
print(f"paid revenue: {paid_by_order['paid_revenue'].sum():,.2f} BRL")

103886 payments → 99440 orders
paid revenue: 16,008,872.12 BRL


## 3. Joins

### Revenue by customer state

`customer_state` lives in `customers`, the revenue in `payments`. A question that needs two
tables needs a `merge`.

In [115]:
orders_with_state = orders.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left",
    validate="one_to_one"
)
assert len(orders_with_state) == len(orders)

In [116]:
orders.head(1)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18


In [117]:
orders_with_state.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,SP


`validate="one_to_one"` writes our grain assumption into the join. If either side had
repeated keys, pandas would stop with an error instead of silently multiplying rows.

Now the paid revenue, already at order grain. `indicator=True` adds a `_merge` column that
says where each row came from.

In [122]:
small_orders = orders[["order_id", "customer_id"]]

In [123]:
small_orders.head()

,order_id,customer_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c


In [118]:
orders_with_paid = orders_with_state.merge(
    paid_by_order, on="order_id", how="left", validate="one_to_one", indicator=True
)
orders_with_paid["_merge"].value_counts()

_merge
both          99440
left_only         1
right_only        0
Name: count, dtype: int64

In [120]:
orders_with_paid[orders_with_paid._merge == "left_only"]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,paid_revenue,_merge
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,SP,NaN,left_only


In [121]:
assert len(orders_with_paid) == len(orders)
revenue_by_state = (
    orders_with_paid.groupby("customer_state")
    .agg(orders=("order_id", "size"), paid_revenue=("paid_revenue", "sum"))
    .sort_values("paid_revenue", ascending=False)
)
assert abs(revenue_by_state["paid_revenue"].sum() - payments["payment_value"].sum()) < 0.01
revenue_by_state.head()

,orders,paid_revenue
customer_state,,
SP,41746,5998226.96
RJ,12852,2144379.69
MG,11635,1872257.26
RS,5466,890898.54
PR,5045,811156.38


São Paulo alone: 42% of the orders, 37% of the revenue. And the last `assert` proves the
states add up to the payments table: nothing lost, nothing added.

### `inner` vs `left`: who disappears?

*Your turn (1 min):* predict how many rows `orders.merge(items, on="order_id")` returns
(inner join, the default). There are 99,441 orders and 112,650 item lines. Store the actual
number in `n_inner`.

In [124]:
n_inner = len(orders.merge(items, on="order_id"))
n_inner

112650

In [125]:
orders_and_items = orders.merge(items, on="order_id", how="left", indicator=True)
print(len(orders.merge(items, on="order_id")), "rows with inner,", len(orders_and_items), "with left")
orders_and_items["_merge"].value_counts()

112650 rows with inner, 113425 with left


_merge
both          112650
left_only        775
right_only         0
Name: count, dtype: int64

In [126]:
orders_without_items = orders_and_items[orders_and_items["_merge"] == "left_only"]
orders_without_items["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

775 orders have no items: mostly unavailable or cancelled. An inner join drops them **without a
word**. Sometimes that's what you want; decide it, don't let the default decide for you.

### The 20.3 M BRL

Sales wants paid revenue *by seller state*. The seller lives in `items`, the money in
`payments`. The obvious join:

In [127]:
items_and_payments = items.merge(payments, on="order_id")
print(len(items), "items,", len(payments), "payments →", len(items_and_payments), "rows")
print(f"'revenue': {items_and_payments['payment_value'].sum():,.2f} BRL")

112650 items, 103886 payments → 117601 rows
'revenue': 20,308,134.71 BRL


**20.3 M.** An order with 2 items and 2 payments becomes 2 × 2 = 4 rows, and each payment
is counted twice. This is a **fan-out**: a many-to-many join on `order_id`. `validate` catches it:

In [128]:
try:
    items.merge(payments, on="order_id", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge


(The `try`/`except` lets the notebook carry on so we can read the error. In your own work, let
it fail.)

Fix attempt: aggregate payments to order grain first, then join.

In [129]:
items_and_paid = items.merge(paid_by_order, on="order_id", validate="many_to_one")
print(f"'revenue': {items_and_paid['paid_revenue'].sum():,.2f} BRL")

'revenue': 20,308,134.71 BRL


The join is valid now, yet the total is **still 20.3 M**. The order total sits on every item
line, so summing it over item lines counts each order once per item.

**The rule:** sum a number only at its own grain. Paid revenue is an order number: sum it over
orders. To split revenue by seller, use the number that lives on the item line, **item revenue**.

*Your turn (3 min):* join `items` to `sellers` (validate the grain!) and compute item revenue
by `seller_state` into `revenue_by_seller_state`, sorted from highest to lowest.

In [130]:
items_with_seller = items.merge(
    sellers[["seller_id", "seller_state"]], on="seller_id", how="left", validate="many_to_one"
)
revenue_by_seller_state = (
    items_with_seller.groupby("seller_state")
    .agg(item_revenue=("item_revenue", "sum"))
    .sort_values("item_revenue", ascending=False)
)
revenue_by_seller_state.head()

,item_revenue
seller_state,
SP,10235883.88
PR,1458900.73
MG,1224159.80
RJ,937814.12
SC,738973.13


### Prove it

In [30]:
items_with_seller = items.merge(
    sellers[["seller_id", "seller_state"]], on="seller_id", how="left", validate="many_to_one"
)
revenue_by_seller_state = items_with_seller.groupby("seller_state").agg(item_revenue=("item_revenue", "sum"))

assert len(items_with_seller) == len(items)
assert abs(revenue_by_seller_state["item_revenue"].sum() - items["item_revenue"].sum()) < 0.01
print(f"item revenue: {items['item_revenue'].sum():,.2f} BRL")
print(f"paid revenue: {payments['payment_value'].sum():,.2f} BRL")

item revenue: 15,843,553.24 BRL
paid revenue: 16,008,872.12 BRL


15.84 M by items, 16.01 M paid. The gap is about 1%: vouchers and instalment interest that sit
in payments but not on the item line. Both numbers are right; they answer different questions.
The 20.3 M answers none.

**For the CFO: 16.0 M BRL**, and the three `assert`s that prove it.

## Today in one table

| Step | Check |
|---|---|
| Write the clean layer | read it back: types survived |
| State the grain | `is_unique`, `duplicated(subset=…)` |
| Change the grain | `groupby(key).agg(name=(col, func))`, then assert the key is unique |
| Join | `validate=`, `indicator=True`, assert the row count |
| Sum | only at the number's own grain; assert totals match the source |